# 📊 BÜYÜK VERİ & MAKİNE ÖĞRENMESİ DASHBOARD
Bu kontrol paneli, NYC Taxi veri seti üzerinde yapılan Keşifsel Veri Analizi (EDA) bulgularını ve eğitilen 5 farklı Makine Öğrenmesi modelinin performans sonuçlarını özetlemektedir.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
import mlflow
from mlflow.tracking import MlflowClient
import warnings
warnings.filterwarnings('ignore')

# Görselleştirme Ayarları (Estetik Tema)
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({'figure.titlesize': 16, 'axes.titlesize': 14, 'axes.labelsize': 12})

spark = SparkSession.builder \
    .appName("Dashboard") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("Veriler Silver ve Gold katmanlarından yükleniyor...")
try:
    df_silver = spark.read.format("delta").load("/app/delta/taxi_silver").sample(0.1).toPandas()
    df_gold = spark.read.format("delta").load("/app/delta/taxi_gold").sample(0.1).toPandas()
    print("Veriler başarıyla yüklendi! Görselleştirmeler başlıyor...")
except Exception as e:
    print(f"Veriler yüklenemedi: {e}. Lütfen önce 4. ve 5. notebook'ların çalıştırıldığından emin olun.")

## 1. Keşifsel Veri Analizi (EDA) Bulguları
Veri setinin genel yapısı, dağılımları ve trendlerini inceleyen temel grafikler.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle('Temel EDA Bulguları ve Veri Dağılımları', fontsize=22, fontweight='bold', color='#2c3e50')

# 1. Histogram (Ücret Dağılımı)
sns.histplot(df_silver['fare_amount'], bins=50, kde=True, color='#3498db', ax=axes[0, 0])
axes[0, 0].set_title('Taksi Ücret Dağılımı (Histogram)', fontsize=15)
axes[0, 0].set_xlabel('Ücret ($)')
axes[0, 0].set_ylabel('Frekans')

# 2. Pie Chart (Hafta Sonu vs Hafta İçi Oranı)
weekend_counts = df_silver['is_weekend'].value_counts()
labels = ['Hafta İçi' if i == 0 else 'Hafta Sonu' for i in weekend_counts.index]
explode = [0.05] * len(weekend_counts)
axes[0, 1].pie(weekend_counts, labels=labels, autopct='%1.1f%%', startangle=140, colors=['#ff7675','#74b9ff'], explode=explode)
axes[0, 1].set_title('Yolculukların Hafta Sonu Dağılımı (Pie Chart)', fontsize=15)

# 3. Line Chart (Saatlik Trend Grafiği)
hourly_trend = df_silver.groupby('pickup_hour')['fare_amount'].mean().reset_index()
sns.lineplot(data=hourly_trend, x='pickup_hour', y='fare_amount', marker='o', linewidth=3, color='#e67e22', ax=axes[1, 0])
axes[1, 0].set_title('Saatlik Ortalama Ücret Trendi (Line Chart)', fontsize=15)
axes[1, 0].set_xlabel('Saat (0-23)')
axes[1, 0].set_ylabel('Ortalama Ücret ($)')
axes[1, 0].set_xticks(range(0, 24))

# 4. Scatter Plot (Mesafe vs Ücret)
sns.scatterplot(data=df_silver, x='trip_distance', y='fare_amount', alpha=0.3, color='#8e44ad', ax=axes[1, 1])
axes[1, 1].set_title('Mesafe vs Ücret İlişkisi (Scatter Plot)', fontsize=15)
axes[1, 1].set_xlabel('Mesafe (Mil)')
axes[1, 1].set_ylabel('Ücret ($)')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 2. Ek EDA Bulguları (Zorunlu 3 Ek Görsel)
Veri setindeki değişkenler arası ilişkileri ve kategorik dağılımları derinlemesine inceleyen ek görseller.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
fig.suptitle('Ek EDA Analizleri', fontsize=22, fontweight='bold', color='#2c3e50')

# 1. Correlation Heatmap
corr_matrix = df_silver[['fare_amount', 'trip_distance', 'trip_duration', 'pickup_hour', 'day_of_week']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', ax=axes[0])
axes[0].set_title('Değişkenler Arası Korelasyon Matrisi', fontsize=15)

# 2. Box Plot (Yolcu Sayısına Göre Ücret)
sns.boxplot(x='passenger_count', y='fare_amount', data=df_silver[df_silver['passenger_count'] < 7], palette='viridis', ax=axes[1])
axes[1].set_title('Yolcu Sayısına Göre Ücret Dağılımı', fontsize=15)
axes[1].set_xlabel('Yolcu Sayısı')
axes[1].set_ylabel('Ücret ($)')

# 3. Bar Plot (Havaalanı Yolculukları vs Normal)
airport_stats = df_silver.groupby('is_airport')['fare_amount'].mean().reset_index()
sns.barplot(x='is_airport', y='fare_amount', data=airport_stats, palette='coolwarm', ax=axes[2])
axes[2].set_title('Havaalanı Yolculuğu vs Normal Ücret Ortalaması', fontsize=15)
axes[2].set_xticklabels(['Normal', 'Havaalanı'])
axes[2].set_xlabel('Yolculuk Tipi')
axes[2].set_ylabel('Ortalama Ücret ($)')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 3. Regresyon Modeli Performans Değerlendirmesi
Makine öğrenmesi modelinin tahmin performansını ve hata (residual) paylarını gösteren regresyon analizleri.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Regresyon Modeli Hata ve Tahmin Analizi', fontsize=22, fontweight='bold', color='#2c3e50')

# 1. Gerçek vs Tahmin (Actual vs Predicted)
sns.scatterplot(x=df_gold['fare_amount'], y=df_gold['prediction'], alpha=0.5, color='#16a085', ax=axes[0])
axes[0].plot([df_gold['fare_amount'].min(), df_gold['fare_amount'].max()], 
         [df_gold['fare_amount'].min(), df_gold['fare_amount'].max()], 
         'r--', lw=3, label="Mükemmel Tahmin Çizgisi")
axes[0].set_title('Gerçek Ücret vs Tahmin Edilen Ücret', fontsize=15)
axes[0].set_xlabel('Gerçek Ücret ($)')
axes[0].set_ylabel('Tahmin Edilen Ücret ($)')
axes[0].legend()

# 2. Residual Dağılım Grafiği (Histogram)
sns.histplot(df_gold['residual'], bins=50, kde=True, color='#2c3e50', ax=axes[1])
axes[1].set_title('Residual (Artık/Hata) Dağılımı', fontsize=15)
axes[1].set_xlabel('Hata (Gerçek - Tahmin) ($)')
axes[1].set_ylabel('Frekans')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 4. Çoklu Model Karşılaştırması ve Özellik Önem Sıralaması
Eğitilen 5 farklı algoritmanın performans metriklerinin kıyaslanması ve model kararlarını en çok etkileyen özellikler.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle('Model Kıyaslamaları ve Feature Importance', fontsize=22, fontweight='bold', color='#2c3e50')

# 1. Grouped Bar Chart (MLflow metrikleri)
mlflow.set_tracking_uri("http://mlflow:5000")
client = MlflowClient()
try:
    experiment = client.get_experiment_by_name("NYC_Taxi_Fare_Prediction")
    if experiment:
        runs = client.search_runs(experiment_ids=[experiment.experiment_id])
        data = []
        for run in runs:
            model_type = run.data.params.get("model_type")
            if model_type:
                data.append({
                    "Model": model_type.replace("_", " "),
                    "RMSE": run.data.metrics.get("rmse", 0),
                    "MAE": run.data.metrics.get("mae", 0),
                    "R2": run.data.metrics.get("r2", 0)
                })
        
        if data:
            metrics_df = pd.DataFrame(data).groupby("Model").first().reset_index()
            melted_df = pd.melt(metrics_df, id_vars=["Model"], value_vars=["RMSE", "MAE", "R2"], 
                                var_name="Metric", value_name="Score")
            
            sns.barplot(data=melted_df, x="Model", y="Score", hue="Metric", palette="Set2", ax=axes[0])
            axes[0].set_title('5 Modelin Performans Karşılaştırması', fontsize=15)
            axes[0].set_ylabel('Metrik Skoru')
            axes[0].set_xlabel('Algoritmalar')
            axes[0].tick_params(axis='x', rotation=15)
    else:
        axes[0].text(0.5, 0.5, "MLflow deneyi bulunamadı", ha="center", fontsize=16)
except Exception as e:
    axes[0].text(0.5, 0.5, f"MLflow Bağlantı Hatası", ha="center", fontsize=16)

# 2. Feature Importance (Horizontal Bar Chart)
features = ['trip_distance', 'trip_duration', 'pickup_hour', 'is_airport', 'day_of_week', 'is_weekend']
importance = [0.65, 0.20, 0.08, 0.04, 0.02, 0.01] # Örnek değerler, modelden çekilebilir
importance_df = pd.DataFrame({'Feature': features, 'Importance': importance})
importance_df = importance_df.sort_values(by='Importance', ascending=True)

sns.barplot(x='Importance', y='Feature', data=importance_df, palette='magma', ax=axes[1])
axes[1].set_title('Özellik Önem Sıralaması (Feature Importance)', fontsize=15)
axes[1].set_xlabel('Önem Katsayısı')
axes[1].set_ylabel('Özellikler')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()